# 01 — Lorenz Data Generation: Every Noise Combination

One system, two regimes, every combination of observational and dynamical noise.

| Regime | ρ | Behaviour |
|--------|---|-----------|
| **Periodic** | 21 | Stable limit cycle |
| **Chaotic** | 28 | Strange attractor |

**Noise levels:** 0%, 5%, 10%, 50%, 100% of the clean signal's range

**The grid:** 5 obs levels × 5 dyn levels = 25 combos per regime

| obs \\ dyn | 0% | 5% | 10% | 50% | 100% |
|-----------|----|----|-----|-----|------|
| **0%** | clean ×5 ICs | dyn only | dyn only | dyn only | dyn only |
| **5%** | obs only | combined | combined | combined | combined |
| **10%** | obs only | combined | combined | combined | combined |
| **50%** | obs only | combined | combined | combined | combined |
| **100%** | obs only | combined | combined | combined | combined |

**Output:** Windows only 5 windows of 2,000 points each.

**Total: 290 files** (145 per regime)

In [3]:
# ─────────────────────────────────────────────────────────────
# load the packages we need
# ─────────────────────────────────────────────────────────────

using DifferentialEquations                 # solves ODEs and SDEs
using DelimitedFiles                        # writes .dat files
using Statistics                            # mean, std, max, min
using Random                                # seed!, randn

Random.seed!(42)      

TaskLocalRNG()

## Block 1 — Imports and Configuration

In [4]:
# ─────────────────────────────────────────────────────────────
# all the settings in one place
# ─────────────────────────────────────────────────────────────

# how long each time series is
const N_TOTAL   = 12000                     # raw points from the solver
const N_DISCARD = 2000                      # throw away this many (transient)
const N_POINTS  = N_TOTAL - N_DISCARD       # what we keep: 10,000 points
const N_WINDOW  = 2000                      # each series gets cut into windows of this size
const DT        = 0.01                      # time step for the ODE/SDE solver

# noise as fractions of the clean signal's range
# 0.0 = no noise, 1.0 = noise amplitude equals the full signal range
const NOISE_LEVELS = [0.0, 0.05, 0.10, 0.50, 1.00]

# the two regimes we study
const REGIMES = Dict(
    "periodic" => 21.0,                     # rho=21: limit cycle
    "chaotic"  => 28.0                      # rho=28: strange attractor
)

# 5 different starting points for the clean runs
# gives the clean category enough data to match the noisy categories
const CLEAN_ICS = [
    [1.0,  1.0,  1.0],                     # standard starting point
    [1.1,  0.9,  1.2],                     # slightly shifted
    [0.5,  1.5,  0.8],                     # different region
    [-1.0, 1.0,  1.0],                     # negative x start
    [2.0,  0.5,  0.5],                     # further out
]

# where all the .dat files go
const DATA_DIR = "data"
mkpath(DATA_DIR)                            # create the folder if it doesn't exist

# print the plan so we know what to expect
n_grid    = length(NOISE_LEVELS)^2          # 5 x 5 = 25 total combos
n_nonzero = n_grid - 1                      # 24 combos where at least one noise > 0
n_clean   = length(CLEAN_ICS)               # 5 clean runs
n_total   = (n_nonzero + n_clean) * (N_POINTS ÷ N_WINDOW) * 2  # x2 for both regimes

println("PLAN:")
println("  noise levels:       $NOISE_LEVELS")
println("  grid:               $(length(NOISE_LEVELS)) x $(length(NOISE_LEVELS)) = $n_grid combos")
println("  non-zero combos:    $n_nonzero per regime")
println("  clean runs (ICs):   $n_clean per regime")
println("  windows per series: $(N_POINTS ÷ N_WINDOW)")
println("  expected files:     $n_total")

PLAN:
  noise levels:       [0.0, 0.05, 0.1, 0.5, 1.0]
  grid:               5 x 5 = 25 combos
  non-zero combos:    24 per regime
  clean runs (ICs):   5 per regime
  windows per series: 5
  expected files:     290


## Block 2 — Lorenz Equations

In [5]:
# ─────────────────────────────────────────────────────────────
# the deterministic part of the Lorenz system
# ─────────────────────────────────────────────────────────────
# this function computes the three derivatives:
#   dx/dt = sigma * (y - x)
#   dy/dt = x * (rho - z) - y
#   dz/dt = x * y - beta * z
#
# du = where we write the derivatives
# u  = current state [x, y, z]
# p  = parameters [sigma, rho, beta, epsilon]
# t  = current time (not used, but the solver needs it)

function lorenz!(du, u, p, t)
    sigma = p[1]                            # always 10
    rho   = p[2]                            # 21 for periodic, 28 for chaotic
    beta  = p[3]                            # always 8/3

    du[1] = sigma * (u[2] - u[1])           # rate of change of x
    du[2] = u[1] * (rho - u[3]) - u[2]      # rate of change of y
    du[3] = u[1] * u[2] - beta * u[3]       # rate of change of z
end


# ─────────────────────────────────────────────────────────────
# the noise part — only used when solving SDEs
# ─────────────────────────────────────────────────────────────
# additive noise: each of the three equations gets a random
# kick of size epsilon at every time step

function lorenz_noise!(du, u, p, t)
    epsilon = p[4]                           # how strong the noise is

    du[1] = epsilon                          # noise on the x equation
    du[2] = epsilon                          # noise on the y equation
    du[3] = epsilon                          # noise on the z equation
end

lorenz_noise! (generic function with 1 method)

## Block 3 — Helper Functions

In [6]:
# ─────────────────────────────────────────────────────────────
# save_ts — write a single time series to a .dat file
# ─────────────────────────────────────────────────────────────
# one number per line, plain text
# load in Python with: data = np.loadtxt("data/filename.dat")

function save_ts(series::Vector{Float64}, filename::String)
    filepath = joinpath(DATA_DIR, filename)  # e.g. "data/chaotic_obs5_dyn10_w1.dat"
    writedlm(filepath, series)               # write to disk
    println("    saved $(length(series)) pts -> $filepath")
end


# ─────────────────────────────────────────────────────────────
# save_windows — cut a 10,000-point series into 5 windows
# ─────────────────────────────────────────────────────────────
# no full file saved — just the 5 windows of 2,000 points each
# the CNN trains on these windows directly
#
# example: base_name = "chaotic_obs5_dyn20"
#   window 1: points     1 to  2000 -> chaotic_obs5_dyn20_w1.dat
#   window 2: points  2001 to  4000 -> chaotic_obs5_dyn20_w2.dat
#   window 3: points  4001 to  6000 -> chaotic_obs5_dyn20_w3.dat
#   window 4: points  6001 to  8000 -> chaotic_obs5_dyn20_w4.dat
#   window 5: points  8001 to 10000 -> chaotic_obs5_dyn20_w5.dat

function save_windows(series::Vector{Float64}, base_name::String)
    n_win = length(series) ÷ N_WINDOW        # 10000 / 2000 = 5 windows
    for w in 1:n_win                          # loop w = 1, 2, 3, 4, 5
        i_start = (w - 1) * N_WINDOW + 1     # first index of this window
        i_end   = w * N_WINDOW               # last index of this window
        window  = series[i_start:i_end]      # slice out 2000 points
        fname   = base_name * "_w" * string(w) * ".dat"  # e.g. "..._w3.dat"
        save_ts(window, fname)               # write to disk
    end
end

save_windows (generic function with 1 method)

## Block 4 — Clean Baseline and Signal Range

One clean run per regime to measure the **range** (max − min). This range defines what "10% noise" means:

- ε = 0.10 × range

So if the chaotic signal swings from −18 to +18 (range = 36), then 10% noise = ε = 3.6.

In [7]:
# ─────────────────────────────────────────────────────────────
# generate one clean run per regime to measure the signal range
# ─────────────────────────────────────────────────────────────

# store the clean series and their ranges for later use
clean_baseline = Dict{String, Vector{Float64}}()
signal_ranges  = Dict{String, Float64}()

for (regime_name, rho_val) in REGIMES
    println("\n== clean baseline: $regime_name (rho=$rho_val) ==")

    # parameters: [sigma, rho, beta] — no epsilon for the clean ODE
    params    = [10.0, rho_val, 8.0/3.0]
    u0        = CLEAN_ICS[1]                 # start from [1, 1, 1]
    time_span = (0.0, N_TOTAL * DT)         # from t=0 to t=120 seconds

    # set up and solve the ODE
    prob = ODEProblem(lorenz!, u0, time_span, params)
    sol  = solve(prob, Tsit5(); saveat=DT)   # Tsit5 = accurate Runge-Kutta

    # extract just the x values from the [x, y, z] solution
    x_raw = [sol.u[i][1] for i in 1:length(sol)]

    # throw away the transient (first 2000 points)
    x_clean = x_raw[(N_DISCARD + 1):end]

    # trim to exactly 10,000 if the solver gave a few extra
    if length(x_clean) > N_POINTS
        x_clean = x_clean[1:N_POINTS]
    end

    # the range is our ruler for calibrating noise
    sig_range = maximum(x_clean) - minimum(x_clean)

    println("  length: $(length(x_clean)) points")
    println("  range:  $(round(sig_range, digits=3))")
    println("  min:    $(round(minimum(x_clean), digits=3))")
    println("  max:    $(round(maximum(x_clean), digits=3))")

    # save for use in later blocks
    clean_baseline[regime_name] = x_clean
    signal_ranges[regime_name]  = sig_range
end


== clean baseline: periodic (rho=21.0) ==
  length: 10000 points
  range:  1.091
  min:    -7.834
  max:    -6.743

== clean baseline: chaotic (rho=28.0) ==
  length: 10000 points
  range:  35.869
  min:    -18.34
  max:    17.529


## Block 5 — Clean Runs (obs=0%, dyn=0%) from 5 Initial Conditions

The (0%, 0%) cell of the grid is just one combo, so it would only produce 5 window files. To balance it against the other combos, we run 5 times from different starting points.

Because Lorenz is chaotic, different starting points → completely different trajectories → genuine diversity.

**5 ICs × 5 windows = 25 files per regime**

In [8]:
# ─────────────────────────────────────────────────────────────
# 5 clean runs per regime, each from a different starting point
# ─────────────────────────────────────────────────────────────

println("\n" * "="^60)
println("CLEAN RUNS (obs=0%, dyn=0%)")
println("="^60)

for (regime_name, rho_val) in REGIMES
    println("\n  -- $regime_name --")

    for (r, u0) in enumerate(CLEAN_ICS)      # r = 1, 2, 3, 4, 5
        # set up the ODE with this starting point
        params    = [10.0, rho_val, 8.0/3.0]
        time_span = (0.0, N_TOTAL * DT)

        prob = ODEProblem(lorenz!, u0, time_span, params)
        sol  = solve(prob, Tsit5(); saveat=DT)

        # extract x, discard transient, trim
        x_raw   = [sol.u[i][1] for i in 1:length(sol)]
        x_clean = x_raw[(N_DISCARD + 1):end]
        if length(x_clean) > N_POINTS
            x_clean = x_clean[1:N_POINTS]
        end

        # filename: chaotic_obs0_dyn0_r1, chaotic_obs0_dyn0_r2, etc.
        tag = regime_name * "_obs0_dyn0_r" * string(r)
        println("\n  run $r | start = $u0")
        save_windows(x_clean, tag)           # saves 5 window files
    end
end


CLEAN RUNS (obs=0%, dyn=0%)

  -- periodic --

  run 1 | start = [1.0, 1.0, 1.0]
    saved 2000 pts -> data/periodic_obs0_dyn0_r1_w1.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r1_w2.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r1_w3.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r1_w4.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r1_w5.dat

  run 2 | start = [1.1, 0.9, 1.2]
    saved 2000 pts -> data/periodic_obs0_dyn0_r2_w1.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r2_w2.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r2_w3.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r2_w4.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r2_w5.dat

  run 3 | start = [0.5, 1.5, 0.8]
    saved 2000 pts -> data/periodic_obs0_dyn0_r3_w1.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r3_w2.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r3_w3.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r3_w4.dat
    saved 2000 pts -> data/periodic_obs0_dyn0_r3_w5.dat

  run 4

## Block 6 — Every (obs%, dyn%) Combination

The main loop. For every pair where at least one noise level is nonzero:

1. **If dyn > 0:** solve the Lorenz SDE (noise inside the equations)
2. **If dyn = 0:** use the clean baseline (no SDE needed)
3. **If obs > 0:** add Gaussian measurement noise on top
4. **If obs = 0:** keep the signal as-is

This naturally covers obs-only, dyn-only, and combined cases in one loop.

**24 combos × 5 windows × 2 regimes = 240 files**

In [9]:
# ─────────────────────────────────────────────────────────────
# loop over every (obs%, dyn%) pair where at least one is > 0
# ─────────────────────────────────────────────────────────────

println("\n" * "="^60)
println("ALL NOISE COMBINATIONS")
println("="^60)

combo_count  = 0                             # track progress
total_combos = (length(NOISE_LEVELS)^2 - 1) * length(REGIMES)  # 24 x 2 = 48

for (regime_name, rho_val) in REGIMES

    # get the clean signal range for this regime (from Block 4)
    sig_range = signal_ranges[regime_name]

    # loop over every obs level
    for obs_frac in NOISE_LEVELS

        # loop over every dyn level
        for dyn_frac in NOISE_LEVELS

            # skip (0, 0) — that's the clean category from Block 5
            if obs_frac == 0.0 && dyn_frac == 0.0
                continue
            end

            # progress counter
            combo_count += 1
            obs_pct = Int(obs_frac * 100)    # e.g. 5
            dyn_pct = Int(dyn_frac * 100)    # e.g. 20

            println("\n  [$combo_count/$total_combos] $regime_name | obs=$(obs_pct)% dyn=$(dyn_pct)%")

            # ── STEP 1: get the base signal ──
            # if there's dynamical noise, we solve the SDE
            # if not, we just use the clean baseline

            if dyn_frac > 0.0
                # compute the dynamical noise amplitude
                eps_dyn = dyn_frac * sig_range

                # parameters: [sigma, rho, beta, epsilon]
                params = [10.0, rho_val, 8.0/3.0, eps_dyn]

                # set up the SDE: drift + diffusion + starting point + time + params
                prob = SDEProblem(
                    lorenz!,                 # deterministic part
                    lorenz_noise!,           # noise part
                    CLEAN_ICS[1],            # start from [1, 1, 1]
                    (0.0, N_TOTAL * DT),     # time span
                    params                   # [sigma, rho, beta, epsilon]
                )

                # solve the SDE
                # SOSRI = good adaptive solver for additive noise
                # maxiters = high so it doesn't quit on heavy noise
                sol = solve(prob, SOSRI(); saveat=DT, maxiters=5_000_000)

                # extract the x-component
                x_raw  = [sol.u[i][1] for i in 1:length(sol)]

                # discard transient and trim
                x_base = x_raw[(N_DISCARD + 1):end]
                if length(x_base) > N_POINTS
                    x_base = x_base[1:N_POINTS]
                end

                println("    dyn: SDE solved, got $(length(x_base)) pts, eps=$(round(eps_dyn, digits=3))")

            else
                # no dynamical noise — just use the clean signal
                x_base = copy(clean_baseline[regime_name])
                println("    dyn: none (using clean baseline)")
            end

            # ── STEP 2: add observational noise on top if needed ──

            if obs_frac > 0.0
                # obs noise amplitude = fraction of the CLEAN range
                # (not the SDE range — keeps calibration consistent)
                eps_obs   = obs_frac * sig_range

                # random Gaussian noise scaled by eps_obs
                obs_noise = eps_obs .* randn(length(x_base))

                # add it on top of whatever we have
                x_final = x_base .+ obs_noise

                println("    obs: added noise, eps=$(round(eps_obs, digits=3))")
            else
                # no observational noise — keep the signal as-is
                x_final = x_base
                println("    obs: none")
            end

            # ── STEP 3: save the 5 windows ──

            # filename format: chaotic_obs5_dyn20
            tag = regime_name * "_obs" * string(obs_pct) * "_dyn" * string(dyn_pct)
            save_windows(x_final, tag)       # saves 5 files: _w1 through _w5
        end
    end
end

println("\n  done: $combo_count combos generated")


ALL NOISE COMBINATIONS

  [1/48] periodic | obs=0% dyn=5%
    dyn: SDE solved, got 10000 pts, eps=0.055
    obs: none
    saved 2000 pts -> data/periodic_obs0_dyn5_w1.dat
    saved 2000 pts -> data/periodic_obs0_dyn5_w2.dat
    saved 2000 pts -> data/periodic_obs0_dyn5_w3.dat
    saved 2000 pts -> data/periodic_obs0_dyn5_w4.dat
    saved 2000 pts -> data/periodic_obs0_dyn5_w5.dat

  [2/48] periodic | obs=0% dyn=10%
    dyn: SDE solved, got 10000 pts, eps=0.109
    obs: none
    saved 2000 pts -> data/periodic_obs0_dyn10_w1.dat
    saved 2000 pts -> data/periodic_obs0_dyn10_w2.dat
    saved 2000 pts -> data/periodic_obs0_dyn10_w3.dat
    saved 2000 pts -> data/periodic_obs0_dyn10_w4.dat
    saved 2000 pts -> data/periodic_obs0_dyn10_w5.dat

  [3/48] periodic | obs=0% dyn=50%
    dyn: SDE solved, got 10000 pts, eps=0.545
    obs: none
    saved 2000 pts -> data/periodic_obs0_dyn50_w1.dat
    saved 2000 pts -> data/periodic_obs0_dyn50_w2.dat
    saved 2000 pts -> data/periodic_obs0_dyn50

## Block 7 — Summary

In [10]:
# ─────────────────────────────────────────────────────────────
# count everything we generated
# ─────────────────────────────────────────────────────────────

all_files = filter(f -> endswith(f, ".dat"), readdir(DATA_DIR))

println("="^60)
println("DATA GENERATION COMPLETE")
println("="^60)
println("  total .dat files: $(length(all_files))")
println()

# count by noise type
n_clean    = count(f -> contains(f, "obs0_dyn0"), all_files)
n_obs_only = count(f -> contains(f, "_dyn0_") && !contains(f, "obs0_dyn0"), all_files)
n_dyn_only = count(f -> contains(f, "obs0_dyn") && !contains(f, "obs0_dyn0"), all_files)
n_combined = length(all_files) - n_clean - n_obs_only - n_dyn_only

println("  clean (0%, 0%):      $n_clean files")
println("  obs-only (obs>0):    $n_obs_only files")
println("  dyn-only (dyn>0):    $n_dyn_only files")
println("  combined (both>0):   $n_combined files")
println()

# count per regime
for regime in ["periodic", "chaotic"]
    n = count(f -> startswith(f, regime), all_files)
    println("  $regime: $n files")
end

println()
println("filename format: {regime}_obs{X}_dyn{Y}_w{N}.dat")
println("  example: chaotic_obs5_dyn50_w3.dat")
println("  = chaotic regime, 5% obs noise + 50% dyn noise, window 3")
println()
println("to load in Python:")
println("  import numpy as np")
println("  x = np.loadtxt('data/chaotic_obs5_dyn50_w3.dat')")

DATA GENERATION COMPLETE
  total .dat files: 290

  clean (0%, 0%):      50 files
  obs-only (obs>0):    40 files
  dyn-only (dyn>0):    40 files
  combined (both>0):   160 files

  periodic: 145 files
  chaotic: 145 files

filename format: {regime}_obs{X}_dyn{Y}_w{N}.dat
  example: chaotic_obs5_dyn50_w3.dat
  = chaotic regime, 5% obs noise + 50% dyn noise, window 3

to load in Python:
  import numpy as np
  x = np.loadtxt('data/chaotic_obs5_dyn50_w3.dat')
